In [127]:
import pandas as pd
import numpy as np
from pathlib import Path
import json

current_dir = Path.cwd()
project_root = current_dir.parents[2]
path_data = project_root / 'DATA_PPMI/Results/MODEL_DATA/DATA_BL_V08.csv'

data = pd.read_csv(path_data,index_col=0)
patnos = data.index.unique().to_list()

updrs3_cols=['NP3SPCH','NP3FACXP','NP3RIGN','NP3RIGRU','NP3RIGLU','NP3RIGRL','NP3RIGLL','NP3FTAPR','NP3FTAPL','NP3HMOVR','NP3HMOVL','NP3PRSPR','NP3PRSPL','NP3TTAPR','NP3TTAPL','NP3LGAGR','NP3LGAGL','NP3RISNG','NP3GAIT','NP3FRZGT','NP3PSTBL','NP3POSTR','NP3BRADY','NP3PTRMR','NP3PTRML','NP3KTRMR','NP3KTRML','NP3RTARU','NP3RTALU','NP3RTARL','NP3RTALL','NP3RTALJ','NP3RTCON']

def HY3_classification(nhy):
    if nhy == 0:
        return 0  # Healthy
    elif nhy  in [1,2]:
        return 1  # Early Stage
    else:
        return 2  # Advanced Stage

def HY4_classification(nhy):
    if nhy == 0:
        return 0  # Healthy
    elif nhy == 1:
        return 1  # Early Stage
    elif nhy == 2:
        return 2  # Moderate Stage
    else:
        return 3  # Advanced Stage


data["STAGE_NHY3"] = data["NHY"].apply(HY3_classification)
data["STAGE_NHY4"] = data["NHY"].apply(HY4_classification)

V08_cols = ['ENRLLRRK2', 'ENRLGBA', 'COHORT_DEFINITION', 'SEX', 'RAWHITE', 'EDUCYRS', 'AGE_AT_VISIT', "STAGE_NHY3", "STAGE_NHY4"]
data_V08 = data[V08_cols+[ 'EVENT_ID',"NHY"]+updrs3_cols].copy()

data_V08 = data_V08.loc[data_V08["EVENT_ID"] == 'V08',:]  # last visit
data_V08.drop(columns=["EVENT_ID"], inplace=True)
data_V08['UPDRS3_TOTAL_V08'] = data_V08[updrs3_cols].sum(axis=1)
data_V08.drop(columns=updrs3_cols, inplace=True)

data_V06 = data[V08_cols+[ 'EVENT_ID',"NHY"]+updrs3_cols].copy()
data_V06['UPDRS3_TOTAL_V06'] = data_V06[updrs3_cols].sum(axis=1)
data_V06.drop(columns=updrs3_cols, inplace=True)
data_V06 = data_V06.loc[data_V06["EVENT_ID"] == 'V06',:]  # actual visit
data_V06.drop(columns=["EVENT_ID"], inplace=True)

print("data_V08:", data_V08.shape)
print("data_V06:", data_V06.shape)

data_V08 = data_V08.merge(
    data_V06[['UPDRS3_TOTAL_V06']],
    left_index=True,
    right_index=True,
    how='inner')

print("data_V08 after merge:", data_V08.shape)
data_V08['Delta_UPDRS3'] = data_V08['UPDRS3_TOTAL_V08'] - data_V08['UPDRS3_TOTAL_V06']

def classification_mcid(x):
    if x <= -3:
        return 0  # Clinically meaningful improvement
    elif x >= 4:
        return 2   # Clinically meaningful worsening
    else:
        return 1   # Stable / no clinically meaningful change

data_V08['MCID_CLASS'] = data_V08['Delta_UPDRS3'].apply(classification_mcid)
data_V08.drop(columns=['UPDRS3_TOTAL_V06','COHORT_DEFINITION'], inplace=True)
data_V08.head(10)



data_V08: (909, 11)
data_V06: (909, 11)
data_V08 after merge: (909, 12)


,ENRLLRRK2,ENRLGBA,SEX,RAWHITE,EDUCYRS,AGE_AT_VISIT,STAGE_NHY3,STAGE_NHY4,NHY,UPDRS3_TOTAL_V08,Delta_UPDRS3,MCID_CLASS
PATNO,,,,,,,,,,,,
3003,0,0,0.0,1.0,16.0,59.7,1,2,2.0,46.0,3.0,1
3018,0,0,0.0,1.0,16.0,63.6,1,2,2.0,38.0,3.0,1
3020,0,0,0.0,1.0,15.0,77.0,2,3,3.0,45.0,9.0,2
3028,0,0,1.0,1.0,22.0,78.8,1,2,2.0,36.0,-13.0,0
3051,0,0,1.0,1.0,18.0,74.7,1,2,2.0,22.0,3.0,1
3054,0,0,1.0,1.0,16.0,77.3,1,2,2.0,24.0,1.0,1
3056,0,0,1.0,1.0,16.0,58.7,1,2,2.0,31.0,-2.0,1
3058,0,0,1.0,1.0,19.0,82.8,2,3,3.0,53.0,9.0,2
3060,0,0,1.0,1.0,16.0,78.1,1,1,1.0,12.0,-2.0,1


In [128]:
other_cols = [col for col in data.columns if col not in V08_cols]
data_removed = data[V08_cols].copy()
data_stats = data[other_cols].copy()
data_stats = data_stats.loc[data_stats["EVENT_ID"].isin(['BL','V04','V06']),:]  # last visit
data_stats.drop(columns=["EVENT_ID"], inplace=True)

df_grouped = data_stats.groupby(level="PATNO")

df_mean = df_grouped.mean().add_suffix("_mean")
df_min  = df_grouped.min().add_suffix("_min")
df_max  = df_grouped.max().add_suffix("_max")
df_var  = df_grouped.var().add_suffix("_var")
df_std  = df_grouped.std().add_suffix("_std")

data_stats = pd.concat([df_mean, df_min, df_max, df_var, df_std], axis=1)
final_data = data_V08.merge(data_stats, left_index=True, right_index=True, how='inner')
print("final_data:", final_data.shape)

final_data: (909, 942)


# Cols x Domain

In [129]:
cols_dict ={'M_data': [], 'NM_data': [], 'SC_data': [], 'DAT_data': []}

## SC

In [130]:
SC_data = pd.read_csv(project_root / 'DATA_PPMI/Results/SUBJECT_CHARACTERISTICS.csv')
SC_data_cols = SC_data.columns.tolist()
SC_data_cols.remove('PATNO')
SC_data_cols.remove('EVENT_ID')
SC_data_cols.remove('COHORT_DEFINITION')
cols_dict['SC_data'] = SC_data_cols

## Motor

In [131]:
M_data = pd.read_csv(project_root / 'DATA_PPMI/Results/MDS_UPDR.csv')

M_data_cols = M_data.columns.tolist()
M_data_cols.remove('PATNO')
M_data_cols.remove('EVENT_ID')

stats_cols_M = []

for col in M_data_cols:
    for suffix in ["_mean", "_min", "_max", "_var", "_std"]:
        stats_cols_M.append(col + suffix)

cols_dict['M_data'] = stats_cols_M

## Non-Motor

In [132]:
NM_data = pd.read_csv(project_root/ 'DATA_PPMI/Results/NON_MOTOR_DATA.csv' )
NM_data_cols = NM_data.columns.tolist()
NM_data_cols.remove('PATNO')
NM_data_cols.remove('EVENT_ID')
stats_cols_NM = []
for col in NM_data_cols:
    for suffix in ["_mean", "_min", "_max", "_var", "_std"]:
        stats_cols_NM.append(col + suffix)

cols_dict['NM_data'] = stats_cols_NM

# DaTScan, uso en Standby reduccion significativa de datos

In [133]:
data_img = pd.read_csv(project_root / 'DATA_PPMI/Results/DatScan_SBR.csv')
data_img = data_img.loc[data_img["PATNO"].isin(patnos) & data_img["EVENT_ID"].isin(['BL', 'V04', 'V06', 'V08']),:]  # last visit
data_img['New_ADQUISITION'] = data_img['EVENT_ID'].map({'BL': 0, 'V04': 1, 'V06': 2, 'V08': 3})
data_img['PATNO'].nunique() # reduccion de pacientes elevada
data_img = data_img.loc[data_img.groupby('PATNO')['New_ADQUISITION'].idxmax()]
data_img.drop(columns=['EVENT_ID', 'New_ADQUISITION'], inplace=True)
print("data_img:", data_img.shape)
cols_dict['DAT_data'] = data_img.drop(columns=['PATNO']).columns.tolist()
data_img.head(10)


data_img: (790, 34)


,PATNO,STRIATUM_REF_CWM,CAUDATE_REF_CWM,PUTAMEN_REF_CWM,PRECAUDATE_REF_CWM,POSCAUDATE_REF_CWM,PRECOMMISSURAL_PUTAMEN_REF_CWM,POSCOMMISSURAL_PUTAMEN_REF_CWM,PREDORSALPUTAMEN_REF_CWM,PREVENTRALPUTAMEN_REF_CWM,...,CAUDATE_R_REF_CWM,PUTAMEN_R_REF_CWM,PRECAUDATE_R_REF_CWM,POSCAUDATE_R_REF_CWM,PRECOMMISSURAL_PUTAMEN_R_REF_CWM,POSCOMMISSURAL_PUTAMEN_R_REF_CWM,PREDORSALPUTAMEN_R_REF_CWM,PREVENTRALPUTAMEN_R_REF_CWM,POSDORSALPUTAMEN_R_REF_CWM,POSVENTRALPUTAMEN_R_REF_CWM
7,3003,0.69,0.81,0.56,0.99,0.25,0.92,0.27,0.90,0.94,...,0.94,0.77,1.13,0.36,1.19,0.40,1.20,1.18,0.35,0.53
14,3018,0.54,0.46,0.58,0.54,0.21,0.85,0.37,0.92,0.78,...,0.53,0.71,0.64,0.19,1.11,0.37,1.15,1.05,0.38,0.31
29,3028,0.64,0.60,0.65,0.73,0.22,0.99,0.38,0.99,0.99,...,0.53,0.53,0.69,0.04,0.77,0.32,0.80,0.73,0.27,0.46
33,3051,0.78,0.79,0.77,0.98,0.21,1.21,0.42,1.27,1.14,...,0.85,0.80,1.03,0.28,1.27,0.39,1.38,1.13,0.39,0.39
37,3054,0.80,0.81,0.75,0.98,0.27,1.10,0.47,1.06,1.15,...,0.84,0.77,1.03,0.23,1.12,0.47,1.00,1.29,0.42,0.63
41,3060,0.49,0.36,0.52,0.46,0.05,0.82,0.29,0.77,0.88,...,0.43,0.45,0.52,0.11,0.72,0.21,0.67,0.80,0.18,0.29
45,3061,1.02,1.16,0.90,1.41,0.39,1.41,0.50,1.43,1.38,...,1.48,1.19,1.76,0.58,1.79,0.66,1.82,1.74,0.68,0.62
52,3067,0.59,0.43,0.63,0.52,0.13,1.05,0.30,1.03,1.08,...,0.48,0.75,0.56,0.22,1.18,0.39,1.14,1.22,0.32,0.60
56,3077,0.37,0.20,0.44,0.26,0.02,0.62,0.29,0.62,0.62,...,0.30,0.54,0.37,0.07,0.73,0.37,0.68,0.80,0.41,0.24
60,3078,0.83,0.97,0.74,1.12,0.51,1.00,0.54,1.08,0.90,...,1.01,1.02,1.15,0.58,1.38,0.71,1.45,1.29,0.70,0.76


# JSON file 

In [135]:
with open(project_root /"SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/Domain_data.json", "w") as archivo:
    json.dump(cols_dict, archivo, indent=4)